Script evaluating the spatial likelihood mapping of ResNet models, comparing a retinotopic version and a Cartesian version of the network. The aim is to analyse the areas of the image where the model assigns the highest probability to the predicted label. We are testing:

    The generation of likelihood heatmaps for each image in the dataset, identifying the regions that maximise or minimise the probability of the correct label.
    Alignment between the model's high-activation regions and the ground truth annotations, in order to assess the relevance of the highlighted areas.
    Addition of localisation metrics such as Pointing Game (PG), Intersection over Union (IoU) and activation ratio.
    The differences in behaviour between the retinotopic and cartesian versions of the network, in order to understand their respective sensitivity to the spatial structure of the images.

The results are stored in Parquet files for further analysis.

In [1]:
from retinotopy import *

In [2]:
# The dataset to import images from

data_set_type = 'full'
args = Params()
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
args.folders = ['val'] # type of images to use

args.do_saccade = True
args.do_resize = False
args.do_mask = False
args.do_polar = False

## doing the computations

In [ ]:
pos_pred_zero = ((args.resolution[0]*args.resolution[1])//2)
retino_grid = get_grid(args).unsqueeze(0).to(device)
print(args)

annotations = None
#for model_data_set_type in data_set_types:
for model_data_set_type in ['bbox', 'raw']:
    print(50*'=')
    print(f'{model_data_set_type=}')    
    
    for model_name in  ['resnet101']:
    #for model_name in  ['resnet18', 'resnet50', 'resnet101']:
        print(50*'.')
        
        for do_polar in [True, False]:

            do_polar = do_polar if model_data_set_type != 'raw' else False

            args.do_polar = do_polar
            print(f'{args.do_polar=}')

            df_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, do_polar) + f'_map_paper.parquet'
            print(df_filename)
            
            if os.path.isfile(df_filename):
                continue
            else:
                df_map = None
                
                print(50*'.')
                
                model_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, do_polar) + '.pt' if model_data_set_type != 'raw' else None

                
                model = load_model(model_name=model_name, model_path=model_filename, do_circular=args.do_polar).to(device).eval()
                
                if annotations is None:
                    annotations = get_annotation('csv')

                with torch.no_grad():
                    image_datasets = image_datasets_transforms(args, verbose=False)['val']

                    for i_image, (images, label) in tqdm(enumerate(image_datasets)):

                        image_name = image_datasets.samples[i_image][0].split('/')[-1].split('.')[0]
                        
                        ground_true_indices, ground_true, origin_size = get_ground_true(args, image_name, annotations, 'Imagenet')
                        
                        if ground_true.min() == 0.0:
                            
                            since = time.time()

                            images = images.to(device, non_blocking=True)

                            heat_map = model(images)

               
    
                            _, pred_zero = torch.max(heat_map[pos_pred_zero].data, 0) # get the predicted label and its likelihood within the all image
    
                            pred_grid = torch.argmax(heat_map, dim=1) # get the most frequent predicted label (top1 and top5)
                            best_pred = torch.argmax(torch.bincount(pred_grid)).item()
                            best_pred_5 = label if label in get_top_indices(torch.bincount(pred_grid), 5) else best_pred
            
                            
            
                            sum = torch.argmax(torch.sum(heat_map, dim=0)).item() # get the label predict when we sum the evidence on all fixation points or inside the box
                            sum_in = torch.argmax(torch.sum(heat_map[ground_true_indices[0]], dim=0)).item()
    
                            
    
                            heatmap_soft = torch.nn.functional.softmax(heat_map, dim=1)
    
                            likelihood_zero = torch.max(heatmap_soft[pos_pred_zero].data).item()
                            
                            heatmap_prior = heatmap_soft[:, label]

    
                            arg_max_prior = torch.argmax(heatmap_prior)
                            position_prior = (arg_max_prior.item()//min(args.resolution), arg_max_prior.item()%min(args.resolution))
                            _, pred_prior = torch.max(heat_map[arg_max_prior].data, 0) # pred at pov_max_prior
                            likelihood_prior = torch.max(heatmap_prior).item()

                            arg_min_prior = torch.argmin(heatmap_prior)
                            position_min = (arg_min_prior.item()//min(args.resolution), arg_min_prior.item()%min(args.resolution))
                            _, pred_prior_min = torch.max(heat_map[arg_min_prior].data, 0) # pred at pov_min_prior
                            likelihood_prior_min = torch.min(heatmap_prior).item()
                        
    
                            arg_max_no_prior = torch.argmax(heat_map)
                            pred_no_prior = (arg_max_no_prior%1000).item()
                            likelihood_no_prior = heatmap_soft[(arg_max_no_prior//1000),pred_no_prior].item()

    
                            out_heat = th_delete(heatmap_prior, ground_true_indices[0])
                            
    
                            likelihood_out_max = torch.max(out_heat).item()
                            likelihood_in_max = torch.max(heatmap_prior[ground_true_indices[0]]).item()
    
                            likelihood_out_mean = torch.mean(out_heat).item()
                            likelihood_in_mean = torch.mean(heatmap_prior[ground_true_indices[0]]).item()
    
                            Iou = get_IoU(heatmap_prior.cpu(), ground_true.reshape(args.resolution[0]*args.resolution[1]))
                                
                            PG = 1 if likelihood_in_max > likelihood_out_max else 0
                            PG_rev = 1 if arg_max_prior.item() in ground_true_indices[0].data else 0 
                            
                
                            df_map_ = pd.DataFrame({'ImageId':image_name, 'likelihood_in_max': likelihood_in_max, 'likelihood_out_max': likelihood_out_max,
                                                 'pred_zero':pred_zero.item(), 'pred_prior':pred_prior.item(), 'pred_prior_min':pred_prior_min.item(), 'pred_no_prior':pred_no_prior, 'best_pred':best_pred, 
                                                 'likelihood_prior':likelihood_prior, 'likelihood_zero':likelihood_zero, 'likelihood_no_prior':likelihood_no_prior, 'best_pred_5':best_pred_5, 
                                                 'likelihood_out_mean': likelihood_out_mean, 'likelihood_in_mean': likelihood_in_mean, 'Iou': [Iou],
                                                 'position_prior':[position_prior], 'position_min':[position_min], 'likelihood_prior_min':likelihood_prior_min, 
                                                 'sum':sum,'sum_in':sum_in, 'time':time.time() - since, 'label':label, 'PG':PG, 'PG_rev':PG_rev})
                            df_map = store_pandas(df_map, df_map_)

                            if torch.cuda.is_available(): torch.cuda.empty_cache()
                            

                    df_map.to_parquet(df_filename)   

Params(datetag='2025-03-06', loader='data/Imagenet_urls_ILSVRC_2016.json', annotations_animal='data/Animal10k_annotations.json', annotations_train='data/LOC_train_solution.csv', annotations_val='data/LOC_val_solution.csv', folders=['val'], tasks=['animal', 'dog', 'cat', 'bird'], image_size=224, num_epochs=10, n_train_stop=0, seed=1998, batch_size=250, batch_size_val=250, lr=5e-05, momentum=0.02, beta2=0, rs_min=0.0, rs_max=-5.0, do_polar=False, do_raw=False, do_translate=False, do_resize=False, do_mask=False, do_scratch=False, do_rotation=False, resolution=(11, 11), size_ratio=0.1, do_saccade=True, do_zoom=False, method='valid', saccade_type='multi', normalize=True, verbose=False)
model_data_set_type='bbox'
..................................................
args.do_polar=True
cached_data/2025-03-06_bbox_resnet101_retino_map_paper.parquet
..................................................
loading .... cached_data/2025-03-06_bbox_resnet101_retino.pt


1it [00:07,  7.04s/it]

## Analyse the data

In [ ]:
def get_pandas(model_data_set_type, model_name):
    df_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, True) + f'_map_paper.parquet'

    print(f'{df_filename=}')
    with open(df_filename, 'r') as csv_file:
        df_ret = pd.read_parquet(df_filename)


    df_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, False) + f'_map_paper.parquet'
    print(f'{df_filename=}')
    with open(df_filename, 'r') as csv_file:
        df = pd.read_parquet(df_filename)

    df_filename = get_filename(data_cache, datetag, 'raw', model_name, False) + f'_map_paper.parquet'
    print(f'{df_filename=}')
    with open(df_filename, 'r') as csv_file:
        df_raw = pd.read_parquet(df_filename)
    
        
    return df, df_ret, df_raw

In [ ]:
fig_width = 15
fontsize= 20

#for model_data_set_type in ['full' 'bbox']:
for model_data_set_type in ['full']:
    print(50*'=')
    print(f'{model_data_set_type=}')    
    for model_name in  ['resnet101']:
    #for model_name in  ['resnet18', 'resnet50', 'resnet101']:
        print(50*'.')
        
        df, df_ret, df_raw = get_pandas(model_data_set_type, model_name)
                
        df_acc = pd.DataFrame({pred: {label: accuracy_score(df_[pred], df_["label"])
                                            for label, df_ in zip(['raw', 'cartesian', 'retinotopic'], [df_raw, df, df_ret])} 
                               for pred in ['pred_zero', 'pred_prior', 'pred_prior_min', 'pred_no_prior', 'best_pred', 'best_pred_5', 'sum', 'sum_in']})
        
        ax = df_acc.T.plot.bar(rot=30, figsize=(fig_width*1.4, fig_width//3), fontsize=fontsize)
        ax.set_ylim(0, 1)
        ax.hlines(xmin=-.5, xmax=7.5, y=1/1000, ls='--', ec='gray', label='chance level')
        # https://matplotlib.org/stable/gallery/lines_bars_and_markers/bar_label_demo.html
        for container in ax.containers: ax.bar_label(container, padding=-50, color='black', fontsize=fontsize, fmt='%.3f', rotation=90)
        plt.legend(bbox_to_anchor=(1.19, .35), loc='lower right', fontsize=fontsize)
        ax.grid(which='both', axis='y')
        for side in ['top', 'right'] :ax.spines[side].set_visible(False)
        ax.set_title(f'Experiment {model_name} on {model_data_set_type} Imagenet data set', size=fontsize)
        ax.set_ylabel('Accuracy', size=fontsize)
        plt.show();

In [ ]:
# data from https://allisonhorst.github.io/palmerpenguins/

import matplotlib.pyplot as plt
import numpy as np

tasks_color = f'Raw-grey,Cartesian-royalblue,Retinotopic-brown'
tasks_color = dict(map(lambda i: i.split('-'), tasks_color.split(',')))

dataset_name = ("Imagenet", "Animal 10k")
mean_accuracy_before_saccade = {
    'Raw': (.79, .955),
    'Cartesian': (.773, .959),
    'Retinotopic': (.729, .947),
}

x = np.arange(len(dataset_name))  # the label locations
width = 0.25  # the width of the bars
multiplier = 0

 

fig, ax = plt.subplots(1, 1, figsize=(20, 6))

for attribute, measurement in mean_accuracy_before_saccade.items():
    offset = width * multiplier
    rects = ax.bar(x + offset, measurement, width, label=attribute, color=tasks_color[attribute])

    multiplier += 1

#ax.legend(['Raw', 'Cartesian', 'Retinotopic'])

mean_accuracy_after_saccade = {
    'Raw': (.977, 1.),
    'Cartesian': (.962, 1.),
    'Retinotopic': (.958, 1.),
}

multiplier = 0

for attribute, measurement in mean_accuracy_after_saccade.items():
    offset = width * multiplier
    rects = ax.bar(x + offset, measurement, width, alpha=.5, edgecolor =tasks_color[attribute], 
                   linewidth = 2, color=tasks_color[attribute], label='After Saccade')

    multiplier += 1



mean_accuracy_pointing_game = {
    'Raw': (.712, .73),
    'Cartesian': (.787, .743),
    'Retinotopic': (.851, .877),
}

#ax.legend(['After saccade'])

multiplier = 0

for attribute, measurement in mean_accuracy_pointing_game.items():
    offset = (width * multiplier) 
    rects = ax.bar(x + offset, measurement, 0.15, alpha=.75, color='green', label='Pointing Game', edgecolor ='white', linewidth = 2, hatch='//')
    ax.bar_label(rects, padding=3, weight='bold', fontsize=fontsize, fmt='%.3f')

    multiplier += 1
#ax.legend(['Pointing Game'])
#plt.legend(bbox_to_anchor=(1.22, .25), loc='lower right', fontsize=10)
ax.set_ylabel(f"Mean accuracy", size=fontsize)
ax.set_xticks(x + width, dataset_name, fontsize=fontsize)

ax.tick_params(labelsize=20)


if False:
    to_save(fig, 'fig-accuracy_pointing_game_saccade')

In [ ]:
for model_data_set_type in data_set_types:
    print(50*'=')
    print(f'{model_data_set_type=}')    
    for model_name in  ['resnet18', 'resnet50', 'resnet101']:
        print(f'{model_name=}')    
        print(50*'.')

        for do_polar in [False, True]:
            
            args.do_polar = do_polar 

            if do_polar and model_data_set_type =='raw':
                break
            print(f'{args.do_polar=}')
            print(50*'.')


            df_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, do_polar) + f'_map_paper.parquet'

            results = pd.read_parquet(df_filename)
            
            print(results['likelihood_in_mean'].mean(), 'likelihood_in_mean')
            print(results['likelihood_out_mean'].mean(), 'likelihood_out_mean')
            print(results['likelihood_in_mean'].mean()/results['likelihood_out_mean'].mean(), 'likelihood ratio')
            print(results['likelihood_in_max'].mean(), 'likelihood_in_max')
            print(results['likelihood_out_max'].mean(), 'likelihood_out_max')
            

    
            mean_Iou = read_IoU(results['Iou'])
            print(mean_Iou[0], 'mean IoU')

            best_Iou = get_best_Iou(results['Iou'], np.argmax(mean_Iou[0]))
            
            for test_tresh in [0, 0.1, 0.2, 0.3, 0.4, 0.5]:
                inpel = 0
                for i, j in enumerate(results['likelihood_in_max']):
                    if best_Iou[i] > test_tresh and results['label'][i] == results['pred_prior'][i]:
                        inpel += 1
                print(inpel/len(results['likelihood_in_max']), f'GT-known : {test_tresh}')
        
            
            
            print(results['PG'].mean(), 'PG')
        


In [ ]:
for model_data_set_type in ['bbox']:
#for model_data_set_type in ['full', 'bbox']:
    print(50*'=')
    print(f'{model_data_set_type=}')    
    for model_name in  ['resnet101']:
    #for model_name in  ['resnet18', 'resnet50', 'resnet101']:
        print(50*'.')
        below, above, ratio = [], [], []
        df, df_ret, df_raw = get_pandas(model_data_set_type, model_name)

        for df_ in [df_raw, df, df_ret]:

            below.append(df_['likelihood_out_mean'].mean())
            above.append(df_['likelihood_in_mean'].mean())
            ratio.append(df_['likelihood_in_mean'].mean()/df_['likelihood_out_mean'].mean())

        weight_counts = {
            "Out": np.array(below),
            "In": np.array(above),
        }
        
        species = (
            f"No retrain\n ratio={ratio[0]:.1f}",
            f"Cartesian\n ratio={ratio[1]:.1f}",
            f"Retinotopic\n ratio={ratio[2]:.1f}",
        )
        
        width = 0.5
        
        fig, ax = plt.subplots(1, 1, figsize=(5, 5))
        plt.tick_params(axis='both', which='major', labelsize=10)
        bottom = np.zeros(3)
        alpha = 1
        for boolean, weight_count in weight_counts.items():
            p = ax.bar(species, weight_count, width, label=boolean, bottom=bottom, alpha=alpha, color=['grey', 'royalblue', 'brown'])
            #bottom += weight_count
            alpha -= .5
        
        ax.set_ylim(0, 1)
        ax.set_title("(A)")
        ax.legend(loc="upper right")
        ax.set_ylabel(f"Mean likelihood", size=15)
        #ax.set_xlabel(f"Ratio In / Out", size=15)
        #ax.axhline(y=0.5, xmin=0, ls='--', color='black', alpha=.5, xmax=5)
        plt.show()
        if False:
            to_save(fig, 'fig-ratio_paper')

In [ ]:
for model_data_set_type in ['full', 'bbox']:
    print(50*'=')
    print(f'{model_data_set_type=}')    
    for model_name in  ['resnet18', 'resnet50', 'resnet101']:
        print(50*'.')
        
        df, df_ret, df_raw = get_pandas(model_data_set_type, model_name)


        fig, ax = plt.subplots(1, 1, figsize=(5, 5))
        plt.tick_params(axis='both', which='major', labelsize=10)
        
        for df_, label, color in zip([df_raw, df, df_ret], ['Raw', 'Cartesian', 'Retinotopic'], ['grey', 'royalblue', 'brown']):
        
            mean_Iou, list_Iou = read_IoU(df_['Iou'])
        
            all_test = np.linspace(0, 1,36)
            ax.plot(all_test[1:-1], list_Iou[1:-1], lw=2, marker='.', label=label, color=color)
            ax.set_xlabel(f"IoU Threshold", size=15)
            ax.set_ylabel(f"Intersection over Union (IoU)", size=15)
            ax.spines['left'].set_position(('axes', -0.01))
            #ax.set_yscale("logit", one_half="1/2", use_overline=True)
            ax.grid(which='both')
            ax.legend()
            for side in ['top', 'right'] :ax.spines[side].set_visible(False)
            #ax.set_title(f'{model_data_set_type}, {model_name}', size=15);
            ax.set_title(f'(B)', size=15);
            if False:
                to_save(fig, 'fig-IoU_paper')

In [ ]:
fig, axs = plt.subplots(1, 2, sharey=True, figsize=(12, 5))

ax = axs[0]

model_data_set_type = 'bbox'
model_name ==  ['resnet101']

below, above, ratio = [], [], []
df, df_ret, df_raw = get_pandas(model_data_set_type, model_name)

for df_ in [df_raw, df, df_ret]:

    below.append(df_['likelihood_out_mean'].mean())
    above.append(df_['likelihood_in_mean'].mean())
    ratio.append(df_['likelihood_in_mean'].mean()/df_['likelihood_out_mean'].mean())

weight_counts = {
    "Out": np.array(below),
    "In": np.array(above),
}

species = (
    f"No retrain",#\n ratio={ratio[0]:.1f}",
    f"Cartesian",#\n ratio={ratio[1]:.1f}",
    f"Retinotopic",#\n ratio={ratio[2]:.1f}",
)

width = 0.5

plt.tick_params(axis='both', which='major', labelsize=10)
bottom = np.zeros(3)
alpha = 1
for boolean, weight_count in weight_counts.items():
    p = ax.bar(species, weight_count, width, label=boolean, bottom=bottom, alpha=alpha, color=['grey', 'royalblue', 'brown'],
                edgecolor =['grey', 'royalblue', 'brown'], linewidth = 2)
    alpha -= .5

ax.set_ylim(0, 1)
ax.set_title("Mean likehood for the label of interest")
ax.legend(loc="upper right")
ax.set_ylabel(f"Mean likelihood", size=15)
ax.set_title(f'(A)', size=15);
x_pos = [0, 1, 2]
for i in range(len(species)):
    fontweight = 'bold' if ratio[i] == max(ratio) else 'normal'
    ax.text(x = x_pos[i], y = below[i] + 0.01, s = f"Ratio:\n{round(ratio[i], 2)}", fontsize=12, color='black', ha='center', fontweight=fontweight)

ax = axs[1]

#plt.tick_params(axis='both', which='major', labelsize=10)

for df_, label, color in zip([df_raw, df, df_ret], ['Raw', 'Cartesian', 'Retinotopic'], ['grey', 'royalblue', 'brown']):

    mean_Iou, list_Iou = read_IoU(df_['Iou'])


    all_test = np.linspace(0, 1,36)
    ax.plot(all_test[1:-1], list_Iou[1:-1], lw=2, marker='.', label=label, color=color)
    ax.set_xlabel(f"IoU Threshold", size=15)
    ax.set_ylabel(f"Intersection over Union (IoU)", size=15)
    ax.spines['left'].set_position(('axes', -0.01))
    #ax.set_yscale("logit", one_half="1/2", use_overline=True)
    ax.grid(which='both')
    ax.legend()
    for side in ['top', 'right'] :ax.spines[side].set_visible(False)
    #ax.set_title(f'{model_data_set_type}, {model_name}', size=15);
    ax.set_title(f'(B)', size=15)
    ax.tick_params(axis='x', labelsize=15)

plt.show()

if False:
    to_save(fig, 'fig-ImageNet_mesure_paper')